In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from os.path import join, dirname
import os
import histogwas.emb_gwas as eg
import glob

# Table 1

In [4]:
tissue_list_cluster = {
    'Skin_Sun_Exposed_Lower_leg': [0, 1, 2, 3, 4, 5, 7],
 'Esophagus_Muscularis': [0, 1, 2, 3, 4, 5, 6, 7, 8],
 'Stomach': [0, 2, 4, 5, 7, 10],
 'Nerve_Tibial': [0, 6],
 'Colon_Transverse': [0, 1, 2, 3, 5, 6, 7, 8],
 'Esophagus_Mucosa': [0, 1, 2, 3, 4, 7, 8],
 'Artery_Tibial': [0, 2, 4, 5],
 'Breast_Mammary_Tissue': [0, 1, 8],
 'Adipose_Subcutaneous': [0, 1, 2, 3, 4, 5, 6, 7, 8],
 'Muscle_Skeletal': [1, 2, 3, 4, 5, 7],
 'Thyroid': [0, 1, 2, 3, 4, 5, 6]}

In [5]:
tissue_colors = {
    'Esophagus_Muscularis': 'lightseagreen',
    'Esophagus_Mucosa': 'mediumaquamarine',
    'Colon_Transverse': 'lightpink',
    'Stomach': 'palevioletred',
    'Skin_Sun_Exposed_Lower_leg': '#D8BFD8',
    'Breast_Mammary_Tissue': 'rosybrown',
    'Thyroid': '#FFDAB9',
    'Adipose_Subcutaneous': '#ADD8E6',
    'Muscle_Skeletal': 'lightsalmon',
    'Artery_Tibial': 'darkorange',
    'Nerve_Tibial': 'orangered',
}

In [6]:
cluster_count = 0
for tissue in tissue_list_cluster.keys():
    cluster_count +=  len(tissue_list_cluster[tissue])

In [7]:
print(f"Total cluster count {cluster_count}")

Total cluster count 68


# snp to gene prediction

In [8]:
from utils import get_snp, get_expression, get_egenes, merge_dfs
from mtgwas import GWAS
from limix_core.util.preprocess import gaussianize
from statsmodels.stats.multitest import multipletests
import gseapy as gp
from mygene import MyGeneInfo

In [9]:
from histogwas.emb_gwas.utils import df_match

In [10]:
def merge_dfs(df1, df2, key1=None, key2=None, left_index=False, right_index=False):


    if key2 is None:
        key2 = key1

    # df1 indexing
    if left_index:
        _df1 = pd.DataFrame({'x': df1.index.values})
    else:
        _df1 = pd.DataFrame({'x': df1[key1]})
    _df1['idx1'] = np.arange(len(_df1))

    # df2 indexing
    if right_index:
        _df2 = pd.DataFrame({'x': df2.index.values})
    else:
        _df2 = pd.DataFrame({'x': df2[key2]})
    _df2['idx2'] = np.arange(len(_df2))

    # match
    _dfm = _df1.merge(_df2, how='inner', on='x')
    idx1 = _dfm['idx1'].values
    idx2 = _dfm['idx2'].values

    return idx1, idx2

def load_covs(covfile):
    dfcov0 = pd.read_csv(covfile, sep='\t').dropna()
    dfcov = pd.get_dummies(dfcov0['DTHHRDY'].astype(int).astype(str))
    dfcov['SEX'] = 1 * (dfcov0['SEX'].values==2)
    dfcov['AGE'] = dfcov0['AGE'].str.split('-').str.get(0).astype(int) + 5
    dfcov['SID'] = dfcov0['SUBJID']
    dfcov = dfcov.set_index('SID')
    return dfcov

In [11]:
# Here I am reading the bed file and creating its anndata
pcfile='/lustre/groups/casale/datasets/gtex/wgs/GTEx_Analysis_2017-06-05_v8_WholeGenomeSeq_838Indiv_Analysis_Freeze.SHAPEIT2_phased.MAF01.pca.eigenvec'
bfile='/lustre/groups/casale/datasets/gtex/wgs/GTEx_Analysis_2017-06-05_v8_WholeGenomeSeq_838Indiv_Analysis_Freeze.SHAPEIT2_phased.MAF02'

# read bfile and pcs
num_pcs = 4
gdata = eg.read_plink(bfile, pcfile, num_pcs=num_pcs)

# read covs
covfile = '/lustre/groups/casale/datasets/gtex/phenotypes/GTEx_Analysis_v8_Annotations_SubjectPhenotypesDS.txt'
dfcov = load_covs(covfile)

Mapping files: 100%|██████████| 3/3 [00:13<00:00,  4.63s/it]


In [13]:
# Reading the hits
outdir = '/lustre/groups/casale/code/users/shubham.chaudhary/output/projects/gtex/pysrc_v2/preprocessing/supplementary/SupplementaryTable_3'
outfile = join(outdir, 'hits.csv')
df_hits = pd.read_csv(outfile,)

In [14]:
def get_covs(gdata, dfcov):

    # read covs
    covfile = '/lustre/groups/casale/datasets/gtex/phenotypes/GTEx_Analysis_v8_Annotations_SubjectPhenotypesDS.txt'
    dfcov = load_covs(covfile)


    dfpc = (gdata.obs - gdata.obs.mean(0)) / gdata.obs.std(0)
    idx1, idx2 = merge_dfs(dfpc, dfcov, left_index=True, right_index=True)
    dfpc = dfpc.iloc[idx1]
    dfcov = dfcov.iloc[idx2]
#     dfcov[['AGE', 'SEX',]]
    dfcovall = pd.concat([dfpc.iloc[:,:4], dfcov[['AGE', 'SEX',]]], axis=1)
    
    
    return dfcovall

In [15]:
df_hits

,Unnamed: 0,snp,chrom,pos,a0,a1,maf,p_value,cluster_i,tissue,color
0,213910,chr1_78663229_G_A_b38,1,78663229,G,A,0.571043,1.638075e-09,4,Esophagus Mucosa,mediumaquamarine
1,213922,chr1_78667706_T_A_b38,1,78667706,T,A,0.571043,1.638075e-09,4,Esophagus Mucosa,mediumaquamarine
2,3020994,chr5_174255970_A_G_b38,5,174255970,G,A,0.856115,3.936964e-10,3,Skin Sun Exposed Lower leg,#D8BFD8
3,4778910,chr9_97771865_A_G_b38,9,97771865,A,G,0.629496,2.047532e-10,4,Thyroid,#FFDAB9
4,4778911,chr9_97772541_A_T_b38,9,97772541,A,T,0.629496,2.047532e-10,4,Thyroid,#FFDAB9
...,...,...,...,...,...,...,...,...,...,...,...
64,4966601,chr10_17381399_G_A_b38,10,17381399,A,G,0.754496,1.386340e-09,2,Adipose Subcutaneous,#ADD8E6
65,4966602,chr10_17381408_T_C_b38,10,17381408,C,T,0.754496,1.386340e-09,2,Adipose Subcutaneous,#ADD8E6
66,4966607,chr10_17381887_G_A_b38,10,17381887,A,G,0.754496,1.386340e-09,2,Adipose Subcutaneous,#ADD8E6
67,4966608,chr10_17382105_G_A_b38,10,17382105,A,G,0.754496,1.386340e-09,2,Adipose Subcutaneous,#ADD8E6


In [18]:
path_all = '/lustre/groups/casale/code/users/shubham.chaudhary/output/projects/gtex/pysrc_v2/preprocessing/stage6/hgwas/retccl/all_tissue_FWER_corrected.csv'
fwer20 = 3.232995e-09
df_all = pd.read_csv(path_all, sep='\t')
df_all = df_all[df_all['p_value']<fwer20]
df_all = df_all.drop(columns = ['Unnamed: 0.1', 'Unnamed: 0'])

chr9_97772921_C_G_b38 2 Thyroid 'rs7030256' <br>
chr5_174255970_A_G_b38 2 Skin_Sun_Exposed_Lower_leg 'rs1432621' <br>
chr1_78663229_G_A_b38 7 Esophagus_Mucosa 'rs3766325' <br>
chr10_17384475_A_C_b38 1 Adipose_Subcutaneous 'rs2770197' <br>

In [20]:
snp_to_snpid = {'chr9_97772921_C_G_b38': 'rs7030256',
'chr5_174255970_A_G_b38' : 'rs1432621',
'chr1_78663229_G_A_b38' : 'rs3766325',
'chr10_17384475_A_C_b38': 'rs2770197'}

In [21]:
def get_enrichment(df_regulated, dfgene, top=50, type_='FDR'):
    print(f"Doing for top {top} gene")
    
    gene_list = df_regulated['gene_name'].to_list()
    if top != 'all':
        gene_list = gene_list[:top]

    background = dfgene.columns.values.astype(str).tolist()
    # Perform pathway enrichment analysis
    enr_bg = gp.enrichr(gene_list=gene_list, gene_sets=['MSigDB_Hallmark_2020'], organism= 'human', background=background, outdir=None)
    
    return enr_bg.results

In [22]:
# load expression data
def get_id_gene_map(tissue):
    import gzip
    file_path = f'/lustre/groups/casale/code/users/shubham.chaudhary/output/projects/gtex/pysrc_v2/emb-gwas/tpm_gene/gene_tpm_2017-06-05_v8_{tissue.lower()}.gct.gz'
    with gzip.open(file_path, 'rt') as f:
        # Read the GCT file into a pandas DataFrame
        data = pd.read_csv(f, skiprows=2, delimiter='\t')
    
    id_gene_map = dict(zip(data['Name'],data['Description']))
    return id_gene_map

In [23]:
def get_expression_normalized(tissue):
    expr_path = f"/lustre/groups/casale/datasets/gtex/phenotypes/GTEx_Analysis_v8_eQTL_expression_matrices/{tissue}.v8.normalized_expression.bed.gz"
    seed = 0
    dfe = pd.read_csv(expr_path, sep="\t", index_col=3)
    dfe = dfe[dfe['#chr'].str.split('chr').str[-1].isin(np.arange(23).astype(str))]
    dfe = dfe.iloc[:,3:].T
    return dfe

In [24]:
id_gene_map = get_id_gene_map(tissue)
dfgene = get_expression_normalized(tissue)
dfgene.columns = dfgene.columns.map(id_gene_map)

In [ ]:
# Here we are doing association of lead variant(in each signal) with gene expression
import warnings
warnings.filterwarnings('ignore')

association_type = [
    'withCovariate', 
                    # 'withoutcovariates'
                   ]
for type_ in association_type:

    outdir = f'/lustre/groups/casale/code/users/shubham.chaudhary/output/projects/gtex/pysrc_v2/preprocessing/supplementary/SupplementaryTable_3_version7/{type_}'
    os.makedirs(outdir, exist_ok=True)


    correction_type = [
                        # 'FDR', 
                       'Bonferroni'
                      ]
    
    for test_correction_type in correction_type:
        result_all_tissue = []
        for tissue in df_all['tissue'].unique():
           


            _ = df_all[df_all['tissue']==tissue]
            chroms = _['chrom'].unique()
            result_tissue= []
            for chrom in chroms:
                temp_ = _[_['chrom']==chrom]
                temp_ = temp_.sort_values(by=['p_value'])
                snp = temp_.iloc[0]['snp']
                maf = temp_.iloc[0]['maf']
                snp_p_value = temp_.iloc[0]['p_value']
                cluster_i = temp_.iloc[0]['cluster_i']



                snp_id = snp
                tissue = tissue
                dfcovall = get_covs(gdata, dfcov)

                # print(f"Before nan removal {len(dfcovall)}")
                dfcovall = dfcovall.dropna()


                snp_df = get_snp(snp_id, gdata)

                dfgene = get_expression(tissue)
                idx1, idx2, idx3 = df_match([dfgene, snp_df, dfcovall])
                dfgene = dfgene.iloc[idx1]
                snp_df = snp_df.iloc[idx2]
                dfcovall = dfcovall.iloc[idx3]
                covariates = np.concatenate([dfcovall.values, np.ones((dfcovall.values.shape[0], 1))], axis=1)

                # covariates = dfcovall.values
                # print(f"After nan removal {len(dfcovall)}")

                df_result = pd.DataFrame(columns = ['p_value'], index=dfgene.columns)

                dfgene  = gaussianize(dfgene.values)
                Y = dfgene
                if type_ == 'withCovariate':
                    gwas = GWAS(np.array(snp_df))
                    gwas = GWAS(Y, F=covariates)
                else:
                    gwas = GWAS(Y)
                # Here I am gaussianizing the expression
                # U = svd(dfgene.values, full_matrices=False)[0]
                # gwas.process(dfgene) 
                gwas.process(np.array(snp_df)) 
                df_result['p_value'] = gwas.getPv().ravel()
                df_result['tissue'] = tissue
                df_result['snp_id'] = snp_id
                df_result['beta'] = gwas.getBetaSNP().ravel()
                df_result['qv'] = multipletests(df_result['p_value'], method='fdr_bh')[1]
                df_result['maf'] = maf
                df_result['cluster_i'] = cluster_i
                df_result = df_result.sort_values(by=['p_value'])
                temp = df_result


                # if test_correction_type == 'fdr':
                #     threshold = 0.05
                #     temp = df_result[df_result['qv']<(threshold)]
                # else:
                #     threshold = (0.05/len(df_result))
                #     temp = df_result[df_result['p_value']<(threshold)]

                if len(temp) >0:
                    _egene = get_egenes(temp['tissue'][0])
                    _egene = _egene[_egene['gene_name'].isin(temp.index.values)][['gene_name', 'gene_chr', 'gene_start', 'gene_end']].set_index('gene_name')
                    idx_l, idx_r = merge_dfs(temp, _egene, left_index=True, right_index=True)
                    temp = temp.iloc[idx_l]
                    _egene = _egene.iloc[idx_r]
                    temp[['gene_chr', 'gene_start', 'gene_end']] = _egene[['gene_chr', 'gene_start', 'gene_end']]

                    result_tissue.append(temp)
            if len(result_tissue)>0:
                print(tissue)
                tissue_snp_to_expr = pd.concat(result_tissue, axis=0)
                tissue_snp_to_expr = tissue_snp_to_expr.sort_values(by=['p_value'])
                if test_correction_type == 'FDR':
                    outfile = join(outdir, 'snpTogeneFDR', f'Topsnp_to_gene_expr_{tissue}.csv')
                else:
                    outfile = join(outdir, 'snpTogeneBonferroni', f'Topsnp_to_gene_expr_{tissue}.csv')
                os.makedirs(dirname(outfile), exist_ok=True)
                tissue_snp_to_expr.to_csv(outfile, sep='\t', index=True)
                result_all_tissue.append(tissue_snp_to_expr)


        # Reading snpTogene
        top_gene_count_to_consider = [50,
                                      # 100, 150, 200
                                     ]
        outfile = join(outdir, f'snpTogene{test_correction_type}', f'Topsnp_to_gene_expr_*.csv')
        for file in glob.glob(outfile):
            for top_gene_count in top_gene_count_to_consider:
                tissue = file.split('/')[-1].split('expr_')[1].split('.csv')[0]
                df_hits = df_all[df_all['tissue'] == tissue]
                print(tissue)
                df_tissue = pd.read_csv(file, sep='\t')
                df_tissue = df_tissue.sort_values(by=['p_value'])
                dfgene = get_expression(tissue)
                df_tissue.rename(columns={'Unnamed: 0': 'gene_name'}, inplace=True)
                df_upregulated = df_tissue[df_tissue['beta'] > 0]
                df_downregulated = df_tissue[df_tissue['beta']<=0]

                print(f"For {tissue} top {top_gene_count} correction type = {test_correction_type}")
                # pathway_upregulated = get_enrichment(df_upregulated, dfgene, top=top_gene_count,type_=test_correction_type)
                # pathway_downregulated = get_enrichment(df_downregulated, dfgene, top=top_gene_count,type_=test_correction_type)
                if len(df_upregulated) >0:

                    pathway_upregulated = get_enrichment(df_upregulated, dfgene, top = top_gene_count, type_=test_correction_type)
                    display(df_upregulated.head())
                    display(pathway_upregulated.head())
                if len(df_downregulated) >0:
                    pathway_downregulated = get_enrichment(df_downregulated, dfgene, top = top_gene_count, type_=test_correction_type)
                    display(df_downregulated.head())

                    display(pathway_downregulated.head())
                # Create an ExcelWriter object
                out_temp = join(outdir, f'{tissue}_{test_correction_type}_output_top_{top_gene_count}_geneConsidered.xlsx')
                with pd.ExcelWriter(out_temp) as writer:
                    # Write each DataFrame to a separate sheet
                    df_hits.to_excel(writer, sheet_name='hits', index=False)
                    df_tissue.to_excel(writer, sheet_name='snp_to_gene', index=False)
                    if len(df_upregulated) >0:
                        pathway_upregulated.to_excel(writer, sheet_name='upregulated', index=False)
                    if len(df_downregulated) >0:
                        pathway_downregulated.to_excel(writer, sheet_name='downregulated', index=False)

Thyroid
Spleen
Skin_Sun_Exposed_Lower_leg
Esophagus_Mucosa
Adipose_Subcutaneous
Thyroid
For Thyroid top 50 correction type = FDR
Doing for top 50 gene


,gene_name,p_value,tissue,snp_id,beta,qv,maf,cluster_i,gene_chr,gene_start,gene_end
1,CENPJ,8.820354e-31,Thyroid,chr9_97772921_C_G_b38,0.693229,1.868592e-27,0.668831,2,chr13,24882284,24922889
4,TPTE2P1,1.870313e-27,Thyroid,chr9_97772921_C_G_b38,0.654771,1.584903e-24,0.668831,2,chr13,24924677,24968487
6,CPAMD8,1.325091e-19,Thyroid,chr9_97772921_C_G_b38,0.511589,8.020586e-17,0.668831,2,chr19,16892947,17026815
7,DTX4,3.129176e-19,Thyroid,chr9_97772921_C_G_b38,0.543535,1.657290e-16,0.668831,2,chr11,59171430,59208509
10,RP11-756A22.7,5.134648e-18,Thyroid,chr9_97772921_C_G_b38,0.533252,1.977773e-15,0.668831,2,chr13,24933006,24936796


,Gene_set,Term,P-value,Adjusted P-value,Old P-value,Old adjusted P-value,Odds Ratio,Combined Score,Genes
0,MSigDB_Hallmark_2020,Hedgehog Signaling,0.008395,0.124047,0,0,17.358333,82.974162,CELSR1;NKX6-1
1,MSigDB_Hallmark_2020,Pancreas Beta Cells,0.013058,0.124047,0,0,13.342949,57.886840,NKX6-1;FOXA2
2,MSigDB_Hallmark_2020,Bile Acid Metabolism,0.039897,0.252680,0,0,6.918333,22.287129,SULT2B1;HSD17B6
3,MSigDB_Hallmark_2020,Estrogen Response Early,0.054382,0.258316,0,0,3.744073,10.901684,SULT2B1;MUC1;CELSR1
4,MSigDB_Hallmark_2020,Myc Targets V2,0.090896,0.345404,0,0,12.154519,29.147031,SLC19A1


Doing for top 50 gene


,gene_name,p_value,tissue,snp_id,beta,qv,maf,cluster_i,gene_chr,gene_start,gene_end
0,FAM163A,4.537380e-31,Thyroid,chr9_97772921_C_G_b38,-0.694002,1.868592e-27,0.668831,2,chr1,179743163,179816183
2,IGHE,1.957715e-30,Thyroid,chr9_97772921_C_G_b38,-0.650482,2.764947e-27,0.668831,2,chr14,105599941,105601728
3,FAM189A1,6.623082e-28,Thyroid,chr9_97772921_C_G_b38,-0.665878,7.015500e-25,0.668831,2,chr15,29120254,29570723
5,PLA2G4F,4.443909e-21,Thyroid,chr9_97772921_C_G_b38,-0.577363,3.138141e-18,0.668831,2,chr15,42139034,42156636
8,HK2,8.583262e-19,Thyroid,chr9_97772921_C_G_b38,-0.541636,4.040809e-16,0.668831,2,chr2,74833988,74893359


...

,gene_name,p_value,tissue,snp_id,beta,qv,maf,cluster_i,gene_chr,gene_start,gene_end
0,RP11-678G14.3,0.000345,Spleen,chr10_80647340_G_GGT_b38,0.429374,0.999514,0.800377,4,chr19,21570822,21587322
1,BTNL8,0.000567,Spleen,chr10_80647340_G_GGT_b38,0.406682,0.999514,0.800377,4,chr5,180899077,180950906
2,KCNH3,0.001423,Spleen,chr10_80647340_G_GGT_b38,0.370250,0.999514,0.800377,4,chr12,49539157,49558294
3,CLEC7A,0.001670,Spleen,chr10_80647340_G_GGT_b38,0.370797,0.999514,0.800377,4,chr12,10116777,10130258
6,HAL,0.005032,Spleen,chr10_80647340_G_GGT_b38,0.337432,0.999514,0.800377,4,chr12,95972662,95996365


,Gene_set,Term,P-value,Adjusted P-value,Old P-value,Old adjusted P-value,Odds Ratio,Combined Score,Genes
0,MSigDB_Hallmark_2020,Allograft Rejection,0.029939,0.898164,0,0,2.521064,8.845403,KRT1;PRF1;CFP;ELANE;CCR2;TLR2;PF4
1,MSigDB_Hallmark_2020,KRAS Signaling Dn,0.137384,0.967208,0,0,2.119818,4.207786,TG;KRT1;MEFV;UPK3B
2,MSigDB_Hallmark_2020,Myogenesis,0.236090,0.967208,0,0,1.558126,2.249223,CFD;MYBPC3;STC2;ITGA7;MYOM2
3,MSigDB_Hallmark_2020,Reactive Oxygen Species Pathway,0.293675,0.967208,0,0,3.230710,3.958534,MPO
4,MSigDB_Hallmark_2020,Bile Acid Metabolism,0.366636,0.967208,0,0,1.615975,1.621445,PIPOX;GNMT


Doing for top 200 gene


,gene_name,p_value,tissue,snp_id,beta,qv,maf,cluster_i,gene_chr,gene_start,gene_end
4,SNHG5,0.002405,Spleen,chr10_80647340_G_GGT_b38,-0.353937,0.999514,0.800377,4,chr6,85660950,85678736
5,RP11-322E11.5,0.003401,Spleen,chr10_80647340_G_GGT_b38,-0.354299,0.999514,0.800377,4,chr18,35443869,35467088
9,SNHG9,0.006665,Spleen,chr10_80647340_G_GGT_b38,-0.318886,0.999514,0.800377,4,chr16,1964959,1965509
11,HAMP,0.007883,Spleen,chr10_80647340_G_GGT_b38,-0.319683,0.999514,0.800377,4,chr19,35280716,35285143
12,AC016739.2,0.008736,Spleen,chr10_80647340_G_GGT_b38,-0.318259,0.999514,0.800377,4,chr2,176200908,176201252


,Gene_set,Term,P-value,Adjusted P-value,Old P-value,Old adjusted P-value,Odds Ratio,Combined Score,Genes
0,MSigDB_Hallmark_2020,Myc Targets V2,0.005651,0.115737,0,0,11.118421,57.549033,NOP2;HSPE1;HSPD1
1,MSigDB_Hallmark_2020,mTORC1 Signaling,0.006808,0.115737,0,0,3.500836,17.467953,STIP1;SDF2L1;HSPA5;MTHFD2;CACYBP;HSPE1;HSPD1
2,MSigDB_Hallmark_2020,Hypoxia,0.019419,0.174088,0,0,2.570502,10.131694,EFNA1;IL6;CSRP2;HSPA5;ACKR3;ALDOB;CP;IER3
3,MSigDB_Hallmark_2020,UV Response Up,0.020481,0.174088,0,0,3.053233,11.871766,STIP1;DNAJB1;IL6;APOM;FKBP4;CXCL2
4,MSigDB_Hallmark_2020,Myc Targets V1,0.030083,0.199799,0,0,5.123077,17.950148,HSPE1;NME1;HSPD1
